# 🎵 Notebook 4: Acoustic Music Genre Response Classification

## 1. Scientific Background & Hypothesis
In plant perception research, various studies (e.g., Bose 1902, Retallack 1973, Telewski 2006) investigated whether acoustic vibrations influence plant physiology:
- **Classical Music:** Harmonious acoustic spectrum, steady dynamics, lower percussive transients.
- **Devotional Music (Mantra / Chants):** Repetitive resonant chanting (e.g., Om mantra at ~136 Hz and rhythmic chanting).
- **Pop / High-Energy Music:** Highly dynamic percussive beats and rapid acoustic pressure transitions.
- **Normal / Silence:** Resting control without intentional acoustic pressure.

### Objectives:
1. Compare Power Spectral Density and spectral energy across musical genres.
2. Train **Histogram Gradient Boosting (GBDT)** and **ElasticNet / L1-regularized Logistic Regression**.
3. Identify which specific frequency bands (Low, Mid, High) correlate with genre-induced responses.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import PartialDependenceDisplay

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv("plant_time_series_features.csv")

# Filter for music categories and normal baseline
genre_map = {
    "normal": "Silence / Normal",
    "music/classical": "Classical Music",
    "music/devotional": "Devotional Music",
    "music/pop": "Pop Music"
}

music_df = df[df["category"].isin(genre_map.keys())].copy()
music_df["genre"] = music_df["category"].map(genre_map)
print("Genre Distribution:")
print(music_df["genre"].value_counts())


## 2. Spectral Power Distribution Across Acoustic Genres
We examine how the power is partitioned into Low, Mid, and High frequency bands for each genre.


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=music_df, x="genre", y="low_freq_power", ax=ax1, palette="Set3")
ax1.set_title("Low Frequency Power (0 - 0.25 Nyq)", fontweight='bold')
ax1.set_xlabel("")
ax1.tick_params(axis='x', rotation=25)

sns.boxplot(data=music_df, x="genre", y="high_freq_power", ax=ax2, palette="Set3")
ax2.set_title("High Frequency Power (0.50 - 1.0 Nyq)", fontweight='bold')
ax2.set_xlabel("")
ax2.tick_params(axis='x', rotation=25)

sns.boxplot(data=music_df, x="genre", y="spectral_entropy", ax=ax3, palette="Set3")
ax3.set_title("Spectral Entropy (Signal Complexity)", fontweight='bold')
ax3.set_xlabel("")
ax3.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


## 3. Train Gradient Boosting Classifier (GBDT)
We train GBDT using session-grouped cross-validation to predict the acoustic genre from the biopotential windows.


In [ ]:
feature_cols = [
    "mean", "std", "var", "range", "rms", "mad", 
    "zcr", "skew", "kurt", "peak_count",
    "low_freq_power", "mid_freq_power", "high_freq_power", 
    "spectral_entropy", "dominant_freq"
]

genres = ["Silence / Normal", "Classical Music", "Devotional Music", "Pop Music"]
genre_to_idx = {g: i for i, g in enumerate(genres)}

X_music = music_df[feature_cols].values
y_music = music_df["genre"].map(genre_to_idx).values
groups_music = music_df["session_id"].values

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)

gbdt_model = HistGradientBoostingClassifier(max_iter=120, random_state=42)
oof_preds_gbdt = np.zeros(len(y_music), dtype=int)

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_music, y_music, groups_music)):
    gbdt_model.fit(X_music[tr_idx], y_music[tr_idx])
    oof_preds_gbdt[te_idx] = gbdt_model.predict(X_music[te_idx])

print("=== Gradient Boosting Classification Report (Music Genres) ===")
print(classification_report(y_music, oof_preds_gbdt, target_names=genres))


## 4. Multi-Genre Normalized Confusion Matrix


In [ ]:
plt.figure(figsize=(8, 6))
cm_genre = confusion_matrix(y_music, oof_preds_gbdt, normalize='true')
sns.heatmap(cm_genre, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=genres, yticklabels=genres)
plt.title("Gradient Boosting: Acoustic Genre Confusion Matrix", fontweight='bold')
plt.ylabel("True Genre")
plt.xlabel("Predicted Genre")
plt.tight_layout()
plt.show()


## 5. Frequency Band & Dynamic Feature Importance
Evaluating the most informative features in genre discrimination.


In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_music)

lr_model = LogisticRegression(max_iter=1000, penalty='l2', C=1.0)
lr_model.fit(X_norm, y_music)

coef_mean = np.mean(np.abs(lr_model.coef_), axis=0)
indices = np.argsort(coef_mean)[::-1]

plt.figure(figsize=(10, 5))
plt.barh(range(len(feature_cols)), coef_mean[indices][::-1], color="#4A148C", alpha=0.85)
plt.yticks(range(len(feature_cols)), [feature_cols[i] for i in indices][::-1])
plt.xlabel("Mean Absolute Coefficient Weight")
plt.title("Feature Discriminative Power Across Acoustic Genres", fontweight='bold')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()


## 6. Multi-Genre Radar Profile Chart
We plot the mean standardized feature fingerprints for each musical category.


In [ ]:
radar_features = ["std", "mad", "zcr", "spectral_entropy", "low_freq_power", "high_freq_power"]
radar_means = music_df.groupby("genre")[radar_features].mean()
radar_norm = (radar_means - radar_means.min()) / (radar_means.max() - radar_means.min() + 1e-9)

labels = radar_features
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette = {"Silence / Normal": "#1976D2", "Classical Music": "#F06292", "Devotional Music": "#7B1FA2", "Pop Music": "#FF9800"}

for genre in genres:
    values = radar_norm.loc[genre].tolist()
    values += values[:1]
    ax.plot(angles, values, label=genre, color=palette[genre], linewidth=2)
    ax.fill(angles, values, color=palette[genre], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontweight='bold')
ax.set_title("Acoustic Genre Feature Radar Profiles", fontweight='bold', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
